# Round Advisor — EDA

Exploration of the harmonized Adaptyv EGFR competition data (rounds 1+2). Run from the repo root with the project venv kernel.

Key questions: how sparse are the labels, what distinguishes binders, and how much does informed selection help?

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.features import load_features

df = load_features()
print(df.shape)
df[['design_id','round','binds','binding_strength','expression','kd','log10_kd']].head()

## Label sparsity and class balance

In [ ]:
print('designs by round:', df['round'].value_counts().to_dict())
print('binders:', int(df['binds'].sum()), 'of', len(df))
print('KD-labeled:', int(df['kd'].notna().sum()))
print('binding_strength:', df['binding_strength'].value_counts().to_dict())
df.groupby('round')[['binds']].sum()

## KD distribution (binders only)

KD is molar; lower = stronger. Plotted as log10(KD).

In [ ]:
sub = df[df['log10_kd'].notna()]
plt.figure(figsize=(6,4))
plt.hist(sub['log10_kd'], bins=20, color='#4C72B0', edgecolor='white')
plt.xlabel('log10(KD / M)'); plt.ylabel('designs'); plt.title(f'KD distribution (n={len(sub)})')
plt.tight_layout(); plt.show()

## What distinguishes binders?

Compare a few features between binders and non-binders. The AF2/ESM metrics (pae_interaction, plddt, iptm, esm_pll) are expected to separate best.

In [ ]:
feats = ['pae_interaction','plddt','iptm','esm_pll','isoelectric_point','gravy','instability_index']
df.groupby('binds')[feats].median()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13,4))
for ax, f in zip(axes, ['pae_interaction','plddt','esm_pll']):
    for label, color in [(False,'#C44E52'),(True,'#55A868')]:
        vals = df[df['binds']==label][f].dropna()
        ax.hist(vals, bins=20, alpha=0.6, color=color, label=('binder' if label else 'non-binder'))
    ax.set_title(f); ax.legend()
plt.tight_layout(); plt.show()

## Active-learning payoff

Pooled backtest: binders found vs designs tested, informed UCB acquisition vs random.

In [ ]:
from src.active_learning import run_backtest
res = run_backtest()
plt.figure(figsize=(7,4))
plt.plot(res.tested, res.informed_hits, '-o', label='informed (UCB)', color='#55A868')
plt.plot(res.tested, res.random_hits, '--o', label='random', color='#8C8C8C')
plt.axhline(res.total_hits, color='k', ls=':', lw=1, label=f'all {res.total_hits} binders')
plt.xlabel('designs tested'); plt.ylabel('binders found'); plt.legend(); plt.title('Active-learning backtest')
plt.tight_layout(); plt.show()
res.summary_at(100)